In [ ]:
import gc
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# 0) Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def tail_mean(x: np.ndarray, k: int) -> float:
    if x.size == 0:
        return float("nan")
    if x.size <= k:
        return float(np.nanmean(x))
    return float(np.nanmean(x[-k:]))


def compute_pog_series(W_ref: float, W_series: np.ndarray) -> np.ndarray:
    if (not np.isfinite(W_ref)) or W_ref <= 1e-8:
        return np.full_like(W_series, np.nan, dtype=np.float32)
    return ((W_ref - W_series) / W_ref).astype(np.float32)


def detection_delay(pog_hat: np.ndarray, tau: float) -> int:
    """1-based first t where pog_hat[t] >= tau; 0 if never."""
    for i, v in enumerate(pog_hat):
        if np.isfinite(v) and v >= tau:
            return i + 1
    return 0


def rankdata_avg_ties(a: np.ndarray) -> np.ndarray:
    """
    Average-rank for ties (Spearman-friendly).
    """
    a = np.asarray(a)
    order = np.argsort(a, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(a), dtype=float)

    # average ties
    sorted_a = a[order]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sorted_a[j + 1] == sorted_a[i]:
            j += 1
        if j > i:
            avg = (i + j) / 2.0
            ranks[order[i:j + 1]] = avg
        i = j + 1
    return ranks


def spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return float("nan")
    rx = rankdata_avg_ties(x[mask])
    ry = rankdata_avg_ties(y[mask])
    c = np.corrcoef(rx, ry)[0, 1]
    return float(c)


def fmt_table5(df: pd.DataFrame) -> str:
    """
    Match screenshot-style formatting:
      b: 2 decimals
      ρ: 3 decimals
      σ_PoGhat: 4 decimals
      FP/6, FN/6: ints
    """
    df2 = df.copy()

    def f2(v): return f"{v:.2f}"
    def f3(v): return f"{v:.3f}"
    def f4(v): return f"{v:.4f}"

    df2["b"] = df2["b"].map(f2)
    df2["ρ"] = df2["ρ"].map(f3)
    df2["σ_PoGhat"] = df2["σ_PoGhat"].map(f4)
    df2["FP/6"] = df2["FP/6"].astype(int)
    df2["FN/6"] = df2["FN/6"].astype(int)

    return df2.to_string(index=False)


# ============================================================
# 1) Table 5 params (paper notation)
# ============================================================

@dataclass
class Table5Params:
    # FL
    K: int = 12
    R: int = 25
    E: int = 1
    B: int = 64
    eta: float = 0.01
    mu: float = 0.9

    # Non-IID
    alpha_dir: float = 0.5

    # Metric vs welfare split
    head: Tuple[int, ...] = (0, 1, 2, 3, 4)
    tail: Tuple[int, ...] = (5, 6, 7, 8, 9)

    # Public leak
    phi: float = 1.0

    # Profiles (caption says “6 gaming profiles”; we use 6 levels as in your code)
    Theta_profiles: Tuple[float, ...] = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)

    # Small benign fraction
    theta_benign: float = 0.10

    # Update-scaling gaming
    s_update: float = 1.75

    # Audit
    b_grid: Tuple[float, ...] = (0.10, 0.25, 0.50)  # audited client fraction
    S: int = 5  # audit resampling trials

    # Summary window + risk threshold
    L: int = 7
    tau: float = 0.05

    seed: int = 42
    root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


P = Table5Params()


# ============================================================
# 2) Data: FashionMNIST, partitions, public head loader, leak subset, per-client tail-eval loader
# ============================================================

def make_dirichlet_partitions(labels: np.ndarray, K: int, alpha_dir: float, rng: np.random.Generator) -> List[List[int]]:
    n_classes = int(labels.max()) + 1
    client_indices: List[List[int]] = [[] for _ in range(K)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue

        proportions = rng.dirichlet(alpha_dir * np.ones(K))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])

        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(K):
        rng.shuffle(client_indices[cid])

    return client_indices


def filter_subset_by_labels(
    base_subset: Subset,
    train_local_set: Subset,
    full_train: datasets.FashionMNIST,
    allowed: Tuple[int, ...],
) -> Subset:
    allowed_set = set(allowed)
    kept = []
    for idx_in_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_local]
        y = int(full_train.targets[global_idx])
        if y in allowed_set:
            kept.append(idx_in_local)
    return Subset(train_local_set, kept)


def split_subset_train_eval(base_subset: Subset, seed: int, eval_ratio: float = 0.2) -> Tuple[Subset, Subset]:
    idxs = list(base_subset.indices)
    rng = np.random.default_rng(seed)
    rng.shuffle(idxs)
    n_eval = int(round(eval_ratio * len(idxs)))
    ev = idxs[:n_eval]
    tr = idxs[n_eval:]
    return Subset(base_subset.dataset, tr), Subset(base_subset.dataset, ev)


def oversample_tail(
    train_subset: Subset,
    train_local_set: Subset,
    full_train: datasets.FashionMNIST,
    tail: Tuple[int, ...],
    factor: int = 2,
) -> ConcatDataset:
    tail_sub = filter_subset_by_labels(train_subset, train_local_set, full_train, tail)
    if len(tail_sub) == 0:
        return ConcatDataset([train_subset])
    return ConcatDataset([train_subset] + [tail_sub for _ in range(factor)])


def prepare_data(P: Table5Params):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_train = datasets.FashionMNIST(root=P.root, train=True, download=True, transform=transform)

    # 60k -> 50k local + 10k public candidates
    n_local = 50_000
    idx_all = np.arange(len(full_train))
    idx_local = idx_all[:n_local]
    idx_public = idx_all[n_local:]

    train_local_set = Subset(full_train, idx_local.tolist())
    targets_all = np.array(full_train.targets)

    # public head val
    head_mask = np.isin(targets_all, np.array(P.head, dtype=int))
    public_head_idx = np.array([i for i in idx_public if head_mask[i]], dtype=int)
    rng = np.random.default_rng(P.seed)
    rng.shuffle(public_head_idx)

    leak_size = int(P.phi * len(public_head_idx))
    leak_idx = public_head_idx[:leak_size]

    public_val_set = Subset(full_train, public_head_idx.tolist())
    public_val_loader = DataLoader(
        public_val_set, batch_size=P.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available()
    )
    leak_subset = Subset(full_train, leak_idx.tolist())

    # dirichlet partitions on local 50k (index space 0..49999)
    train_targets_local = np.array(full_train.targets)[idx_local]
    client_indices = make_dirichlet_partitions(
        labels=train_targets_local,
        K=P.K,
        alpha_dir=P.alpha_dir,
        rng=np.random.default_rng(P.seed)
    )
    client_base = [Subset(train_local_set, idxs) for idxs in client_indices]

    # per-client train split + tail-eval loaders
    client_train: List[Subset] = []
    client_tail_eval_loaders: List[Optional[DataLoader]] = []

    for cid in range(P.K):
        tr, ev = split_subset_train_eval(client_base[cid], seed=P.seed + 123 + cid, eval_ratio=0.2)
        client_train.append(tr)

        tail_ev = filter_subset_by_labels(ev, train_local_set, full_train, P.tail)
        if len(tail_ev) == 0:
            client_tail_eval_loaders.append(None)
        else:
            client_tail_eval_loaders.append(
                DataLoader(tail_ev, batch_size=P.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
            )

    cleanup()
    return full_train, train_local_set, client_train, client_tail_eval_loaders, public_val_loader, leak_subset


# ============================================================
# 3) Model + local train
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_model(device: torch.device) -> nn.Module:
    return SimpleCNN().to(device)


def local_train(model: nn.Module, dataset, P: Table5Params) -> None:
    loader = DataLoader(dataset, batch_size=P.B, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    opt = optim.SGD(model.parameters(), lr=P.eta, momentum=P.mu)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(P.E):
        for x, y in loader:
            x = x.to(P.device, non_blocking=True)
            y = y.to(P.device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()


def transmit_state(global_sd_cpu: Dict[str, torch.Tensor], local_sd: Dict[str, torch.Tensor], scale: float) -> Dict[str, torch.Tensor]:
    # tx = global + scale*(local-global), computed on CPU
    out = {}
    for k in global_sd_cpu.keys():
        out[k] = global_sd_cpu[k] + scale * (local_sd[k].detach().to("cpu") - global_sd_cpu[k])
    return out


# ============================================================
# 4) Strategies and assignment (same spirit as your original E1 code)
# ============================================================

STR_HONEST = "HONEST"
STR_BENIGN = "BENIGN"
STR_GAME_HEAD = "GAME_HEAD"
STR_GAME_SCALE = "GAME_SCALE"


def assign_strategies(P: Table5Params, theta_gaming: float, rng: np.random.Generator) -> Dict[int, str]:
    ids = np.arange(P.K)
    rng.shuffle(ids)

    n_benign = int(round(P.theta_benign * P.K))
    n_gaming = int(round(theta_gaming * P.K))

    benign_ids = ids[:n_benign]
    gaming_ids = ids[n_benign:n_benign + n_gaming]
    honest_ids = ids[n_benign + n_gaming:]

    strat: Dict[int, str] = {}
    for cid in honest_ids:
        strat[int(cid)] = STR_HONEST
    for cid in benign_ids:
        strat[int(cid)] = STR_BENIGN

    half = len(gaming_ids) // 2
    for cid in gaming_ids[:half]:
        strat[int(cid)] = STR_GAME_HEAD
    for cid in gaming_ids[half:]:
        strat[int(cid)] = STR_GAME_SCALE

    return strat


# ============================================================
# 5) Run one profile -> M_t series and tail_acc_matrix[t,c]
# ============================================================

def fedavg_round_streaming(
    global_sd_cpu: Dict[str, torch.Tensor],
    strat_map: Dict[int, str],
    client_train: List[Subset],
    full_train,
    train_local_set,
    leak_subset,
    P: Table5Params,
) -> Dict[str, torch.Tensor]:
    sum_sd = {k: torch.zeros_like(v, device="cpu") for k, v in global_sd_cpu.items()}

    for cid in range(P.K):
        base_train = client_train[cid]
        strat = strat_map[cid]

        if strat == STR_HONEST:
            ds = base_train
            tx_scale = 1.0

        elif strat == STR_BENIGN:
            ds = oversample_tail(base_train, train_local_set, full_train, P.tail, factor=2)
            tx_scale = 1.0

        elif strat == STR_GAME_HEAD:
            head_only = filter_subset_by_labels(base_train, train_local_set, full_train, P.head)
            ds = ConcatDataset([head_only, leak_subset])
            tx_scale = 1.0

        elif strat == STR_GAME_SCALE:
            ds = base_train
            tx_scale = P.s_update

        else:
            raise ValueError(strat)

        model = make_model(P.device)
        model.load_state_dict(global_sd_cpu, strict=True)
        local_train(model, ds, P)
        local_sd = model.state_dict()

        tx_sd_cpu = transmit_state(global_sd_cpu, local_sd, scale=tx_scale)

        with torch.no_grad():
            for k in sum_sd.keys():
                sum_sd[k] += tx_sd_cpu[k]

        del model, local_sd, tx_sd_cpu
        cleanup()

    with torch.no_grad():
        for k in sum_sd.keys():
            sum_sd[k] /= float(P.K)

    cleanup()
    return sum_sd


def run_profile(P: Table5Params, theta: float, data_bundle, seed_offset: int) -> Dict[str, np.ndarray]:
    full_train, train_local_set, client_train, client_tail_eval_loaders, public_val_loader, leak_subset = data_bundle

    rng = np.random.default_rng(P.seed + seed_offset)
    strat_map = assign_strategies(P, theta, rng)

    # init global state on CPU
    global_model = make_model(P.device)
    global_sd_cpu = {k: v.detach().to("cpu").clone() for k, v in global_model.state_dict().items()}
    del global_model
    cleanup()

    M_hist = np.zeros(P.R, dtype=np.float32)
    tail_acc_matrix = np.full((P.R, P.K), np.nan, dtype=np.float32)

    for r in range(P.R):
        global_sd_cpu = fedavg_round_streaming(
            global_sd_cpu, strat_map, client_train,
            full_train, train_local_set, leak_subset, P
        )

        model_eval = make_model(P.device)
        model_eval.load_state_dict(global_sd_cpu, strict=True)

        # M_t (server-visible head metric)
        M_hist[r] = float(accuracy(model_eval, public_val_loader, P.device))

        # tail eval per client (GT ingredient)
        for cid in range(P.K):
            loader = client_tail_eval_loaders[cid]
            if loader is None:
                continue
            tail_acc_matrix[r, cid] = float(accuracy(model_eval, loader, P.device))

        del model_eval
        cleanup()

    del global_sd_cpu
    cleanup()
    return {"M_hist": M_hist, "tail_acc_matrix": tail_acc_matrix}


# ============================================================
# 6) Audit estimator + Table 5 aggregation
# ============================================================

def audit_estimate_welfare_series(tail_acc_matrix: np.ndarray, b: float, rng: np.random.Generator) -> np.ndarray:
    R, K = tail_acc_matrix.shape
    m = max(1, int(round(b * K)))
    W_hat = np.zeros(R, dtype=np.float32)
    all_ids = np.arange(K)
    for t in range(R):
        audited = rng.choice(all_ids, size=m, replace=False)
        W_hat[t] = float(np.nanmean(tail_acc_matrix[t, audited]))
    return W_hat


def run_table_5(P: Table5Params) -> pd.DataFrame:
    set_seed(P.seed)
    data_bundle = prepare_data(P)

    # Baseline aligned run for W_ref (theta=0, with benign fraction like your E1)
    base = run_profile(P, theta=0.0, data_bundle=data_bundle, seed_offset=0)
    W_ref_series = np.nanmean(base["tail_acc_matrix"], axis=1).astype(np.float32)
    M_ref_series = base["M_hist"].astype(np.float32)
    W_ref = tail_mean(W_ref_series, P.L)
    M_ref = tail_mean(M_ref_series, P.L)

    # Run all profiles once (GT PoG and per-b audit stats)
    profiles_gt_pog: Dict[float, float] = {}
    profiles_tail_mat: Dict[float, np.ndarray] = {}

    seed_offset = 10
    for theta in P.Theta_profiles:
        prof = run_profile(P, theta=float(theta), data_bundle=data_bundle, seed_offset=seed_offset)
        seed_offset += 1

        tail_mat = prof["tail_acc_matrix"]
        W_full_series = np.nanmean(tail_mat, axis=1).astype(np.float32)
        pog_gt_series = compute_pog_series(W_ref, W_full_series)
        pog_gt = tail_mean(pog_gt_series, P.L)

        profiles_gt_pog[float(theta)] = float(pog_gt)
        profiles_tail_mat[float(theta)] = tail_mat  # keep for audits

        # cleanup per profile object (but keep matrix)
        del prof, W_full_series, pog_gt_series
        cleanup()

    # Aggregate per audit budget b
    rows = []
    n_profiles = len(P.Theta_profiles)  # should be 6 (matches caption “/6”)

    for b in P.b_grid:
        pog_hat_means = []
        pog_hat_stds = []
        pred_risky_list = []
        gt_risky_list = []

        # For Spearman: pair (PoG_GT, mean(PoG_hat)) across profiles
        pog_gt_list = []
        pog_hat_mean_list = []

        for theta in P.Theta_profiles:
            theta = float(theta)
            tail_mat = profiles_tail_mat[theta]
            pog_gt = profiles_gt_pog[theta]

            pog_gt_list.append(pog_gt)
            gt_risky = int(np.isfinite(pog_gt) and pog_gt >= P.tau)
            gt_risky_list.append(gt_risky)

            # audit resampling trials
            trial_vals = []
            trial_risky = []
            for s in range(P.S):
                rng_a = np.random.default_rng(P.seed + 1000 + s + int(100 * b) + int(1000 * theta))
                W_hat_series = audit_estimate_welfare_series(tail_mat, b=float(b), rng=rng_a)
                pog_hat_series = compute_pog_series(W_ref, W_hat_series)
                pog_hat = tail_mean(pog_hat_series, P.L)
                trial_vals.append(pog_hat)
                trial_risky.append(int(np.isfinite(pog_hat) and pog_hat >= P.tau))

            mean_hat = float(np.nanmean(trial_vals))
            std_hat = float(np.nanstd(trial_vals))
            pog_hat_means.append(mean_hat)
            pog_hat_stds.append(std_hat)
            pog_hat_mean_list.append(mean_hat)

            # majority vote classification
            pred_risky = int((np.mean(trial_risky) >= 0.5))
            pred_risky_list.append(pred_risky)

            del W_hat_series, pog_hat_series
            cleanup()

        rho = spearman_corr(np.array(pog_gt_list), np.array(pog_hat_mean_list))
        sigma_poghat = float(np.nanmean(np.array(pog_hat_stds, dtype=float)))  # mean of per-profile stds

        FP = int(np.sum([(gt == 0 and pr == 1) for gt, pr in zip(gt_risky_list, pred_risky_list)]))
        FN = int(np.sum([(gt == 1 and pr == 0) for gt, pr in zip(gt_risky_list, pred_risky_list)]))

        rows.append({
            "b": float(b),
            "ρ": float(rho),
            "σ_PoGhat": float(sigma_poghat),
            "FP/6": FP,   # counts out of 6 profiles
            "FN/6": FN,
        })

    # cleanup big objects
    del data_bundle, base, W_ref_series, M_ref_series, profiles_tail_mat
    cleanup()

    df = pd.DataFrame(rows).sort_values("b").reset_index(drop=True)
    return df


if __name__ == "__main__":
    df = run_table_5(P)
    print("Table 5: Reliability of audit-based estimation under partial audits.")
    print(fmt_table5(df))